In [1]:
import numpy as np
from tqdm import tqdm

# ─── SCORE PARSING ────────────────────────────────────────────────────────────

def parse_score(score_str):
    result = []
    for s in score_str.strip().split():
        p1g, p2g = map(int, s.split('-'))
        result.append((p1g, p2g))
    return result

def is_set_complete(p1g, p2g):
    if p1g == 7 and p2g == 6: return True
    if p2g == 7 and p1g == 6: return True
    if p1g >= 6 and p1g - p2g >= 2: return True
    if p2g >= 6 and p2g - p1g >= 2: return True
    return False

def parse_match_state(score_str, best_of):
    sets_won = [0, 0]
    current_set_games = None
    for p1g, p2g in parse_score(score_str):
        if is_set_complete(p1g, p2g):
            if p1g > p2g: sets_won[0] += 1
            else:         sets_won[1] += 1
        else:
            current_set_games = (p1g, p2g)
            break
    sets_needed = best_of // 2 + 1
    assert sets_won[0] < sets_needed and sets_won[1] < sets_needed, \
        "Match is already over according to the score string."
    return sets_won, current_set_games

_NOTATION = {'0': 0, '15': 1, '30': 2, '40': 3, 'Ad': 4, 'AD': 4, 'A': 4}

def parse_game_score(game_score_str, is_tiebreak=False):
    """Convert score string → (p1_pts, p2_pts). Empty/0-0 → (0, 0).
    Regular game: tennis notation ("40-15"). Tiebreak: raw counts ("3-2")."""
    s = game_score_str.strip()
    if not s or s == "0-0":
        return (0, 0)
    left, right = s.split('-')
    if is_tiebreak:
        return (int(left), int(right))
    return (_NOTATION[left], _NOTATION[right])

# ─── POINT / GAME / TIEBREAK / SET ────────────────────────────────────────────

def sim_point(p1_serving):
    """Returns True if P1 wins the point."""
    server, opp = (P1, P2) if p1_serving else (P2, P1)
    p_win_1st = (server['win_first']  + (1 - opp['return_first']))  / 2
    p_win_2nd = (server['win_second'] + (1 - opp['return_second'])) / 2
    if np.random.random() < server['first_in']:
        server_won = np.random.random() < p_win_1st
    else:
        server_won = np.random.random() < p_win_2nd
    return server_won if p1_serving else not server_won


def sim_game(p1_serving, start_score=(0, 0)):
    """
    Returns True if P1 wins the game.
    start_score = (p1_pts, p2_pts) in raw counts (0=0, 1=15, 2=30, 3=40, 4=Ad).
    """
    score = list(start_score)   # score[0]=P1, score[1]=P2
    while True:
        p1_won = sim_point(p1_serving)
        score[0 if p1_won else 1] += 1
        if score[0] >= 4 and score[0] - score[1] >= 2: return True
        if score[1] >= 4 and score[1] - score[0] >= 2: return False


def sim_tiebreak(p1_serves_first, start_score=(0, 0)):
    """Returns True if P1 wins the tiebreak. start_score = (p1_pts, p2_pts)."""
    score = list(start_score)
    point_count = score[0] + score[1]
    while True:
        p1_serves = p1_serves_first if point_count == 0 else p1_serves_first == (point_count % 2 == 0)
        p1_won = sim_point(p1_serves)
        score[0 if p1_won else 1] += 1
        point_count += 1
        if score[0] >= 7 and score[0] - score[1] >= 2: return True
        if score[1] >= 7 and score[1] - score[0] >= 2: return False


def sim_set(p1_serving, start_games=(0, 0), first_game_score=(0, 0)):
    """
    Simulate (the rest of) one set.
    first_game_score = (p1_pts, p2_pts); only applies to the first game/tiebreak played.
    """
    games = list(start_games)
    first_game = True

    while True:
        score = first_game_score if first_game else (0, 0)
        first_game = False

        if games[0] == 6 and games[1] == 6:
            p1_won_tb = sim_tiebreak(p1_serving, score)
            games[0 if p1_won_tb else 1] += 1
            return (games[0] > games[1]), p1_serving

        p1_won_game = sim_game(p1_serving, score)
        games[0 if p1_won_game else 1] += 1
        p1_serving = not p1_serving

        if games[0] == 6 and games[1] == 6:
            p1_won_tb = sim_tiebreak(p1_serving)
            games[0 if p1_won_tb else 1] += 1
            return (games[0] > games[1]), p1_serving

        if games[0] >= 6 and games[0] - games[1] >= 2: return True,  p1_serving
        if games[1] >= 6 and games[1] - games[0] >= 2: return False, p1_serving

# ─── MATCH FROM STATE ─────────────────────────────────────────────────────────

def sim_match_from_state(sets_won, current_set_games, first_game_score, p1_serving, best_of):
    sets_needed = best_of // 2 + 1
    sets = list(sets_won)

    first_set_games = current_set_games if current_set_games is not None else (0, 0)
    p1_won_set, p1_serving = sim_set(p1_serving, first_set_games, first_game_score)
    sets[0 if p1_won_set else 1] += 1

    while sets[0] < sets_needed and sets[1] < sets_needed:
        p1_won_set, p1_serving = sim_set(p1_serving)
        sets[0 if p1_won_set else 1] += 1

    return sets[0] > sets[1]


In [32]:
# ─── INPUTS ───────────────────────────────────────────────────────────────────

P1Name = "cobolli"
P2Name = "zverev"

# P1 = {
#     'first_in':      112/175,
#     'win_first':     85/112,
#     'win_second':    31/63,
#     'return_first':  14/73,
#     'return_second': 29/72,
# }

# P2 = {
#     'first_in':      74/146,
#     'win_first':     60/74,
#     'win_second':    43/72,
#     'return_first':  27/112,
#     'return_second': 32/63,
# }
P1 = {
    'first_in':      0.52,
    'win_first':     0.66,
    'win_second':    0.50,
    'return_first':  0.26,
    'return_second': 0.57,
}

P2 = {
    'first_in':      0.75,
    'win_first':     0.74,
    'win_second':    0.43,
    'return_first':  0.34,
    'return_second': 0.50,
}

BEST_OF = 5
# Space-separated set scores. Completed sets first, then optional in-progress set.
# Examples:  "6-1 2-3"     (set 1 done, set 2 at 2-3)
#            "6-1 6-4 3-2" (two sets done, third at 3-2)
#            "6-1"         (only first set done, second not started yet)
SCORE_STRING = "1-6 6-4 4-6 7-6 0-0"
# True if P1 serves the next/current game
P1_SERVES_NEXT = True
# Current game score (P1-P2). Use "" or "0-0" for start of a new game.
# Regular game: tennis notation e.g. "40-15", "30-30", "Ad-40"
# Tiebreak (set score 6-6): raw point count e.g. "3-2", "6-5"
GAME_SCORE = "0-0"

N = 10_000

# ─── RUN ──────────────────────────────────────────────────────────────────────

sets_won, current_set_games = parse_match_state(SCORE_STRING, BEST_OF)
in_tiebreak = current_set_games == (6, 6)
first_game_score = parse_game_score(GAME_SCORE, is_tiebreak=in_tiebreak)

completed = [f"{a}-{b}" for a, b in parse_score(SCORE_STRING) if is_set_complete(a, b)]
in_prog   = f"{current_set_games[0]}-{current_set_games[1]}" if current_set_games else "not started"
game_disp = GAME_SCORE.strip() if GAME_SCORE.strip() else "0-0"

print(f"{P1Name} vs {P2Name}  |  Best of {BEST_OF}")
print(f"Sets won: {P1Name} {sets_won[0]}  {P2Name} {sets_won[1]}")
print(f"Completed sets: {completed}   Current set: {in_prog}   Current game: {game_disp} (P1-P2){'  [TIEBREAK]' if in_tiebreak else ''}")
print(f"{'P1' if P1_SERVES_NEXT else 'P2'} serves next.  Running {N:,} simulations...\n")

wins = sum(
    sim_match_from_state(sets_won, current_set_games, first_game_score, P1_SERVES_NEXT, BEST_OF)
    for _ in tqdm(range(N))
)

p1_prob = wins / N
p2_prob = 1 - p1_prob
se = np.sqrt(p1_prob * p2_prob / N)

print(f"\n{P1Name} win probability: {p1_prob:.4f}  ±{se:.4f}  95% CI ({p1_prob - 1.96*se:.4f}, {p1_prob + 1.96*se:.4f})")
print(f"{P2Name} win probability: {p2_prob:.4f}  ±{se:.4f}  95% CI ({p2_prob - 1.96*se:.4f}, {p2_prob + 1.96*se:.4f})")


cobolli vs zverev  |  Best of 5
Sets won: cobolli 2  zverev 2
Completed sets: ['1-6', '6-4', '4-6', '7-6']   Current set: 0-0   Current game: 0-0 (P1-P2)
P1 serves next.  Running 10,000 simulations...



100%|██████████| 10000/10000 [00:00<00:00, 20720.71it/s]


cobolli win probability: 0.2501  ±0.0043  95% CI (0.2416, 0.2586)
zverev win probability: 0.7499  ±0.0043  95% CI (0.7414, 0.7584)


In [33]:
from collections import Counter

# ── Current set win probability ───────────────────────────────────────────────
cur_games = current_set_games if current_set_games is not None else (0, 0)
set_wins = sum(
    sim_set(P1_SERVES_NEXT, cur_games, first_game_score)[0]
    for _ in range(N)
)
set_p1 = set_wins / N
print(f"Current set win probability:  {P1Name} {set_p1:.4f}  |  {P2Name} {1-set_p1:.4f}\n")

# ── Full-match scoreline breakdown ────────────────────────────────────────────
def sim_match_score(sets_won, current_set_games, first_game_score, p1_serving, best_of):
    sets_needed = best_of // 2 + 1
    sets = list(sets_won)
    first_set_games = current_set_games if current_set_games is not None else (0, 0)
    p1_won_set, p1_serving = sim_set(p1_serving, first_set_games, first_game_score)
    sets[0 if p1_won_set else 1] += 1
    while sets[0] < sets_needed and sets[1] < sets_needed:
        p1_won_set, p1_serving = sim_set(p1_serving)
        sets[0 if p1_won_set else 1] += 1
    return (sets[0], sets[1])

scoreline_counts = Counter(
    sim_match_score(sets_won, current_set_games, first_game_score, P1_SERVES_NEXT, BEST_OF)
    for _ in tqdm(range(N))
)

sets_needed = BEST_OF // 2 + 1
print(f"Final scoreline probabilities ({P1Name} vs {P2Name}):")
for lost in range(sets_needed):          # P1 wins
    key = (sets_needed, lost)
    if key[0] >= sets_won[0] and key[1] >= sets_won[1]:
        print(f"  {P1Name} wins {key[0]}-{key[1]}: {scoreline_counts.get(key,0)/N:.4f}")
for lost in range(sets_needed):          # P2 wins
    key = (lost, sets_needed)
    if key[0] >= sets_won[0] and key[1] >= sets_won[1]:
        print(f"  {P2Name} wins {key[1]}-{key[0]}: {scoreline_counts.get(key,0)/N:.4f}")

Current set win probability:  cobolli 0.2564  |  zverev 0.7436



100%|██████████| 10000/10000 [00:00<00:00, 20751.86it/s]

Final scoreline probabilities (cobolli vs zverev):
  cobolli wins 3-2: 0.2534
  zverev wins 3-2: 0.7466
